In [ ]:
# Installs to get OPUS-MT
!pip install transformers sacrebleu sentencepiece

In [ ]:
# Mount drive to retrieve datasets
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Load Europarl test set — same split as IBM Model 1
fr_path = '/content/drive/MyDrive/cs505_data/europarl-v7.fr-en.fr'
en_path = '/content/drive/MyDrive/cs505_data/europarl-v7.fr-en.en'

with open(fr_path, encoding='utf-8') as f:
    fr_lines = f.readlines()
with open(en_path, encoding='utf-8') as f:
    en_lines = f.readlines()

TRAIN_SIZE = 50000
TEST_SIZE  = 2000

test_fr = [s.strip() for s in fr_lines[TRAIN_SIZE:TRAIN_SIZE + TEST_SIZE]]
test_en = [s.strip() for s in en_lines[TRAIN_SIZE:TRAIN_SIZE + TEST_SIZE]]

print(f"Test sentences: {len(test_fr):,}")
print(f"Sample FR: {test_fr[0]}")
print(f"Sample EN: {test_en[0]}")

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-fr-en"
model = MarianMTModel.from_pretrained(model_name)
tokenizer = MarianTokenizer.from_pretrained(model_name)
print("Model loaded successfully")

In [ ]:
from tqdm import tqdm
import sacrebleu

# BLEU Eval
def translate_opus(sentences, batch_size=32):
    translations = []
    for i in tqdm(range(0, len(sentences), batch_size)):
        batch = sentences[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
        translated = model.generate(**inputs)
        decoded = [tokenizer.decode(t, skip_special_tokens=True) for t in translated]
        translations.extend(decoded)
    return translations

translations_opus = translate_opus(test_fr)

bleu = sacrebleu.corpus_bleu(translations_opus, [test_en])
print(f"\nOPUS-MT Transformer BLEU score: {bleu.score:.2f}")

OPUS-MT Transformer BLEU score: 37.99

BLEU Score=32.53 on Europarl Test Split

Leaving here if outputs don't save;
Training took more than an hour.

In [ ]:
# Tests

print("\nSample translations:")
for i in range(5):
    print(f"\nFR:  {test_fr[i]}")
    print(f"REF: {test_en[i]}")
    print(f"OPUS: {translations_opus[i]}")

